# IR + Raman + H/C NMR → HOMO–LUMO gap

按 checkpoint 的 ir-raman-nmr 完整输入运行；同时报告训练均值常量基线。

本 notebook 展示已记录的真实评测结果。模型推理需显式开启，始终不进行训练。

In [1]:
import json, os, sys, subprocess, tempfile
from pathlib import Path
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'openmolspec_bench').is_dir()), None)
if root is None:
    raise RuntimeError('Run this notebook from the cloned OpenMolSpec frontend repository')
task_id = 'vibrational.gap.ir_raman_hc'
example_input = {'ir': '1024 min-max-normalized intensities', 'raman': '1024 min-max-normalized intensities', 'wavenumber_cm1': [400, 4000], 'nmr_h_shifts_ppm': [1.2, 3.65], 'nmr_c_shifts_ppm': [18.2, 58.3], 'nmr_reference_id': 'must match checkpoint', 'formula': 'not supplied to Vib2Mol'}
print(json.dumps(example_input, indent=2, ensure_ascii=False))

{
  "ir": "1024 min-max-normalized intensities",
  "raman": "1024 min-max-normalized intensities",
  "wavenumber_cm1": [
    400,
    4000
  ],
  "nmr_h_shifts_ppm": [
    1.2,
    3.65
  ],
  "nmr_c_shifts_ppm": [
    18.2,
    58.3
  ],
  "nmr_reference_id": "must match checkpoint",
  "formula": "not supplied to Vib2Mol"
}


## 已记录的评测

保留样本量、输入条件和来源文件；流程执行成功不等于精度达标。

In [2]:
index = json.loads((root / 'documentation/assets/evaluation_index.json').read_text())
recorded = index.get(task_id, [])
print(json.dumps(recorded or {'status': 'not_evaluated', 'reason': 'Required compatible assets or adapter unavailable'}, indent=2, ensure_ascii=False))

[
  {
    "task": "vibrational.gap.ir_raman_hc",
    "dataset_name": null,
    "status": "passed",
    "metrics": {
      "mae_ev": 0.1314457535089349,
      "rmse_ev": 0.20596364382966384,
      "examples": 128
    },
    "given_nmr": true,
    "source_counts": {
      "ZINCV2": 58,
      "SDBS": 4,
      "PCQM4MV2": 20,
      "WILEYIR": 9,
      "ZINCV1": 12,
      "RXN": 6,
      "QM9": 18,
      "WILEYRAMAN": 1
    },
    "selection": {
      "read": 20390,
      "duplicate_structure": 3662,
      "eligible_distinct_structures": 16728,
      "selected": 128
    },
    "sampling": "reservoir",
    "seed": 42,
    "training_mean_baseline": {
      "constant_ev": 5.645122750946045,
      "mae_ev": 0.8999401965027582,
      "rmse_ev": 1.0696170714220543,
      "examples": 128
    },
    "source_file": "gap_protocol_random128_20261116.json",
    "sample_limit": 128
  }
]


## 可选：冻结权重推理

设置 `OPENMOLSPEC_RUN_NOTEBOOK_INFERENCE=1` 并配置工作区和模型环境。缺少权重时跳过或报错，不会启动训练。

In [3]:
if os.environ.get('OPENMOLSPEC_RUN_NOTEBOOK_INFERENCE') == '1':
    runtime = os.environ.get('OPENMOLSPEC_PYTHON', sys.executable)
    workspace = os.environ.get('OPENMOLSPEC_BENCH_ROOT', str(root.parent))
    with tempfile.TemporaryDirectory(prefix='openmolspec-notebook-') as directory:
        command = [runtime, '-m', 'openmolspec_bench', 'run', '--workspace', workspace, '--tasks', task_id, '--limit', '2', '--output', str(Path(directory) / 'result.json')]
        subprocess.run(command, cwd=root, check=True)
        print((Path(directory) / 'result.json').read_text())
else:
    print('Model execution disabled; recorded evaluation displayed above. No training performed.')

Model execution disabled; recorded evaluation displayed above. No training performed.
